# Dia 06 — JSON

**🃏 Cartas:** 6 · 27 · 48 &nbsp;·&nbsp; **📘 Original:** *Day 6. JSON*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 6 — O CNPJ do rodapé

Terça-feira, 8h50. Na parede do NFA, ao lado do mapa de BH, já tem um post-it novo: **"CEP = Praça
Sete. Endereço copiado."** A Iara bate com a caneta no rodapé da vitrine, logo acima do endereço.

— Agora o CNPJ. **48.217.093/0001-30.** Loja que mostra CNPJ no rodapé parece séria. A vítima vê o
número e relaxa. Quase ninguém confere.

Você abre a BrasilAPI de novo, pronto para consultar, mas ela segura sua mão.

— Calma. Lembra do Dia 2? Antes de perguntar a qualquer servidor, pergunte ao **dígito
verificador**. Se o número nem é um CNPJ possível, você não gasta consulta, não deixa seu IP em log
nenhum e já tem a resposta.

O validador roda em menos de um segundo. O final deveria ser outro. **O CNPJ do rodapé não existe.**
Não é empresa irregular, não é empresa baixada: é um número inventado.

— Isso vai para a ficha — diz a Iara. — E a ficha vai ser em **JSON**, porque daqui a duas semanas
outro script vai ler tudo isso sozinho. Ah, e quando for consultar um CNPJ de verdade, use o do
Banco do Brasil para treinar. A resposta da API é um labirinto de chaves, listas e sócios. Aprenda a
andar nele antes de precisar correr.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-06-json")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Entender a estrutura de um JSON: **objetos** (`{}` → dicionário) e **listas** (`[]` → lista).
- Rodar os exemplos originais: ler um campo e ler o mesmo campo de toda uma lista.
- Navegar na resposta de CNPJ da BrasilAPI com `.get()`, inclusive no quadro de sócios (`qsa`).
- Validar um CNPJ (numérico e alfanumérico) **antes** de consultar.
- Salvar e reler uma ficha de evidências com `json.dump` / `json.load`.

## 🧠 Conceito em 2 minutos

**JSON** (*JavaScript Object Notation*) é o formato que quase toda API usa para devolver dados. Ele
tem só seis tipos, e cada um vira um tipo do Python:

| No JSON | No Python | Exemplo |
|---|---|---|
| objeto `{ }` | `dict` | `{"uf": "MG"}` |
| lista `[ ]` | `list` | `["a", "b"]` |
| texto `"…"` | `str` | `"ATIVA"` |
| número | `int` / `float` | `1999.0` |
| `true` / `false` | `True` / `False` | |
| `null` | `None` | |

Para chegar num dado, você "anda" pelo caminho: `dados["qsa"][0]["nome_socio"]` = *a lista `qsa` →
o primeiro item → o campo `nome_socio`*. Se algum degrau faltar, colchetes quebram com `KeyError`
ou `IndexError`; por isso usamos `dados.get("qsa", [])`.

O JSON **não conhece** datas, `Decimal` nem conjuntos. Na hora de salvar, converta: data vira texto
ISO (`"2025-10-12"`), dinheiro vira texto (`"54877.40"`), conjunto vira lista.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/read_one_field.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-06-json/original/read_one_field.py) | Mostra o total de resultados e o link do primeiro perfil da busca no GitHub |
| [`original/read_list_of_fields.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-06-json/original/read_list_of_fields.py) | Percorre a lista `items` e imprime o link de cada perfil |

```bash
pip install requests
cd dia-06-json
python original/read_one_field.py
python original/read_list_of_fields.py
```

> 💡 Repare no `len(...) - 1` do segundo script: ele faz o laço **pular o último perfil**. Não mexemos
> no original (é a regra), mas é um ótimo exercício: descubra por quê e corrija numa cópia sua.
> O autor também recomenda ferramentas online como <https://jsonpathfinder.com> para descobrir o
> "caminho" até um campo num JSON grande.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q requests

### `original/read_list_of_fields.py` — Ler o mesmo campo de todos os itens de uma lista JSON

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler o mesmo campo de todos os itens de uma lista JSON
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_6/read_list_of_fields.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests
# ──────────────────────────────────────────────────────────────────────────

import json
import requests

response = requests.get("https://api.github.com/search/users?q=javascript")

json_data=response.json()

# 💬 len() conta quantos itens vieram na lista (a API manda 30 por página).
# 💬 Atenção: o "-1" faz o laço pular o ÚLTIMO item. range(n) já vai de 0 até n-1,
# 💬 então bastaria len(json_data['items']). É um bom exemplo de erro "por um" (off-by-one).
usersCount = len(json_data['items'])-1

# 💬 x vale 0, 1, 2... e usamos x como posição na lista para imprimir cada perfil.
for x in range(usersCount):
    print (json_data['items'][x]['html_url'])

### `original/read_one_field.py` — Ler um campo de um JSON (total de resultados e o primeiro perfil)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler um campo de um JSON (total de resultados e o primeiro perfil)
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_6/read_one_field.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests
# ──────────────────────────────────────────────────────────────────────────

# 💬 json vem com o Python; requests precisa de pip install.
# 💬 (Curiosidade: aqui o json nem chega a ser usado, porque o requests já converte.)
import json
import requests

# 💬 O mesmo pedido do Dia 5: usuários do GitHub que combinam com "javascript".
response = requests.get("https://api.github.com/search/users?q=javascript")

# 💬 .json() transforma o texto da resposta num dicionário Python.
json_data=response.json()

# 💬 Colchetes com o nome do campo: pega o valor de "total_count" (quantos usuários existem).
print (json_data['total_count'])

# 💬 Caminho em três passos: a lista "items" → o item de posição 0 (o primeiro) → o campo "html_url".
print (json_data['items'][0]['html_url'])

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Cada script tem amostra offline em `dados/`.

| Arquivo | O que faz |
|---|---|
| [`br/01_cnpj_brasilapi.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-06-json/br/01_cnpj_brasilapi.py) | Consulta o CNPJ público do Banco do Brasil e navega por razão social, situação, CNAE, município e sócios |
| [`br/02_caso_cnpj_rodape.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-06-json/br/02_caso_cnpj_rodape.py) | **O caso:** valida o CNPJ do rodapé antes de consultar; inválido → registra a pista, sem gastar consulta |
| [`br/03_ficha_json.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-06-json/br/03_ficha_json.py) | Monta a ficha de evidências, salva em `saidas/ficha_evidencias.json` e confere a releitura |

```bash
cd dia-06-json
python br/01_cnpj_brasilapi.py
OSINT_OFFLINE=1 python br/02_caso_cnpj_rodape.py     # força o modo offline
python br/03_ficha_json.py
```

**O que você deve ver no caso:** `48.217.093/0001-30` **falha no dígito verificador**. Não é uma
empresa irregular: o número simplesmente não pode existir.

> 🆕 **CNPJ alfanumérico.** A partir de julho de 2026 a Receita Federal começou a emitir CNPJs com
> **letras** nas 12 primeiras posições (os dois dígitos verificadores continuam numéricos). O cálculo
> é o mesmo módulo 11, com cada caractere valendo `ord(c) - 48` (`"0"`→0, `"A"`→17). Os scripts deste
> dia já aceitam os dois formatos. Os CNPJs antigos continuam valendo.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_cnpj_brasilapi.py` — Navegar num JSON de verdade: a ficha de um CNPJ na BrasilAPI.

In [ ]:
%cd -q {PASTA_NB}
"""
Navegar num JSON de verdade: a ficha de um CNPJ na BrasilAPI.

Carta 27 (rank B) — Jutsu do Mapa do JSON

O que você aprende:
  • a diferença entre dicionário {chave: valor} e lista [item, item] num JSON;
  • a usar .get("campo", "padrão") para não quebrar quando um campo falta;
  • a percorrer uma lista de dicionários (o quadro de sócios, "qsa");
  • a listar as chaves de um JSON desconhecido antes de sair escrevendo código.

Alvo de demonstração: o CNPJ 00.000.000/0001-91 (Banco do Brasil), um cadastro público
de empresa estatal. Dados de CNPJ são públicos por lei, mas sócios pessoas físicas
continuam tendo direitos pela LGPD: veja o README.

Como rodar (dentro da pasta dia-06-json):
    python br/01_cnpj_brasilapi.py
    OSINT_OFFLINE=1 python br/01_cnpj_brasilapi.py
"""

import json
import os
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
CNPJ = "00.000.000/0001-91"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── 1. Consultar ──────────────────────────────────────────────────────────
numeros = "".join(c for c in CNPJ if c.isdigit())        # a API quer só os 14 dígitos
dados = buscar_json(f"https://brasilapi.com.br/api/cnpj/v1/{numeros}", "brasilapi_cnpj_00000000000191.json")

# ── 2. Primeiro, reconhecer o terreno: que chaves vieram? ─────────────────
# Diante de um JSON novo, a primeira coisa é olhar as chaves. Isso evita "chutar" nomes.
chaves = sorted(k for k in dados if not k.startswith("_"))   # ignoramos o "_aviso" da amostra
print(f"🗝️  O JSON tem {len(chaves)} chaves no primeiro nível. Algumas: {', '.join(chaves[:8])}…\n")

# ── 3. Campos simples (texto e número) ────────────────────────────────────
# dados["campo"] quebra (KeyError) se o campo não existir; dados.get("campo", "—") não quebra.
print(f"🏢 Razão social:     {dados.get('razao_social', '—')}")
print(f"🏷️  Nome fantasia:    {dados.get('nome_fantasia') or '(sem nome fantasia)'}")
print(f"📌 Situação:         {dados.get('descricao_situacao_cadastral', '—')}")
print(f"🗓️  Início atividade: {dados.get('data_inicio_atividade', '—')}")
print(f"🏭 CNAE principal:   {dados.get('cnae_fiscal', '—')} · {dados.get('cnae_fiscal_descricao', '—')}")
print(f"⚖️  Natureza:         {dados.get('natureza_juridica', '—')}")
print(f"📍 Município/UF:     {dados.get('municipio', '—')}/{dados.get('uf', '—')}")

# Números vêm como número mesmo (float): dá para formatar em reais.
capital = dados.get("capital_social")
if isinstance(capital, (int, float)):
    # Truque para o padrão brasileiro: formata em inglês e troca os separadores.
    em_reais = f"{capital:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"💰 Capital social:   R$ {em_reais}")

# ── 4. Lista de dicionários: o quadro de sócios e administradores (QSA) ───
# "qsa" é uma LISTA. Cada item é um DICIONÁRIO com nome_socio, qualificacao_socio etc.
socios = dados.get("qsa", [])                 # se não vier, usamos lista vazia
print(f"\n👥 QSA: {len(socios)} pessoa(s)/empresa(s)")
for posicao, socio in enumerate(socios, start=1):
    print(f"   {posicao}. {socio.get('nome_socio', '?'):<22} → {socio.get('qualificacao_socio', '?')}")

# ── 5. Lista dentro do JSON: atividades secundárias ───────────────────────
secundarias = dados.get("cnaes_secundarios", [])
print(f"\n🏭 Atividades secundárias: {len(secundarias)}")
for cnae in secundarias[:3]:                  # [:3] = só as três primeiras
    print(f"   • {cnae.get('codigo')} {cnae.get('descricao')}")

### `br/02_caso_cnpj_rodape.py` — Caso Vitrine Fantasma: o CNPJ do rodapé da loja existe?

In [ ]:
%cd -q {PASTA_NB}
"""
Caso Vitrine Fantasma: o CNPJ do rodapé da loja existe?

Carta 27 (rank B) — Jutsu do Mapa do JSON (aplicação no caso)

O que você aprende:
  • a VALIDAR um dado antes de gastar uma consulta (dígito verificador do CNPJ, Dia 2);
  • que a validação vale também para o novo CNPJ alfanumérico (letras A–Z nas 12 primeiras posições);
  • a registrar a pista num pequeno dicionário que vira JSON (amanhã ele cresce).

Como rodar (dentro da pasta dia-06-json):
    python br/02_caso_cnpj_rodape.py
    OSINT_OFFLINE=1 python br/02_caso_cnpj_rodape.py
"""

import json
import os
import re
import urllib.error
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:                              # Windows sem tzdata
    FUSO_BR = timezone(timedelta(hours=-3))


def cnpj_valido(cnpj):
    """Confere os dois dígitos verificadores. Aceita CNPJ numérico e alfanumérico.

    Cada caractere vale ord(c) - 48: '0'→0 … '9'→9, 'A'→17, 'B'→18 …
    Os pesos e o "módulo 11" são os mesmos de sempre.
    """
    limpo = re.sub(r"[^0-9A-Za-z]", "", cnpj).upper()      # tira . / -
    if not re.fullmatch(r"[0-9A-Z]{12}[0-9]{2}", limpo) or len(set(limpo)) == 1:
        return False

    def digito(base, pesos):
        soma = sum((ord(c) - 48) * p for c, p in zip(base, pesos))
        resto = soma % 11
        return "0" if resto < 2 else str(11 - resto)

    d1 = digito(limpo[:12], [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    d2 = digito(limpo[:12] + d1, [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    return limpo[12:] == d1 + d2


# ── 1. Pegar o CNPJ do rodapé da vitrine salva ────────────────────────────
html = (PASTA_CASO / "vitrine_megaoferta.html").read_text(encoding="utf-8")
achado = re.search(r"CNPJ\s*([0-9A-Z]{2}\.?[0-9A-Z]{3}\.?[0-9A-Z]{3}/?[0-9A-Z]{4}-?\d{2})", html)
cnpj = achado.group(1) if achado else "48.217.093/0001-30"
print(f"🏷️  CNPJ no rodapé da loja: {cnpj}")

# ── 2. Validar ANTES de consultar ─────────────────────────────────────────
# Consulta custa: tempo, cota de API e um registro do seu IP lá no servidor.
# Se o número nem é um CNPJ possível, não há o que perguntar.
valido = cnpj_valido(cnpj)
pista = {
    "pista": "CNPJ do rodapé da vitrine",
    "valor": cnpj,
    "fonte": "caso/dados/vitrine_megaoferta.html (rodapé)",
    "digito_verificador_ok": valido,
    "consultado_na_api": False,
    "registrado_em": datetime.now(FUSO_BR).isoformat(timespec="seconds"),
}

if not valido:
    print("🚩 Dígito verificador INVÁLIDO: esse número não pode ser de nenhuma empresa.")
    print("   Não gastamos consulta. Um CNPJ inventado é, por si só, uma pista forte de fraude.")
    pista["conclusao"] = "CNPJ inexistente (falha no dígito verificador): provavelmente inventado."
else:
    # Só chega aqui se o número for possível. Mesmo assim pode não existir (HTTP 404).
    pista["consultado_na_api"] = True
    url = f"https://brasilapi.com.br/api/cnpj/v1/{re.sub(r'[^0-9A-Za-z]', '', cnpj)}"
    if OFFLINE:
        print("⚠️ MODO OFFLINE: CNPJ válido, mas não há amostra do caso; consulta pulada.")
        pista["conclusao"] = "Válido; consultar a API quando houver internet."
    else:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(req, timeout=15) as resposta:
                empresa = json.load(resposta)
            pista["conclusao"] = f"Existe: {empresa.get('razao_social')} ({empresa.get('descricao_situacao_cadastral')})"
        except urllib.error.HTTPError as erro:
            pista["conclusao"] = "Não encontrado na base (HTTP 404)." if erro.code == 404 else f"Erro HTTP {erro.code}"
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}).")
            pista["conclusao"] = "Válido; consulta falhou, tentar de novo."

# ── 3. Prova de que o validador funciona (testes de controle) ─────────────
# Antes de acusar um número de falso, testamos o validador com casos conhecidos.
# NUNCA "conserte" o CNPJ do suspeito para achar o "certo": o resultado pode ser
# uma empresa real e inocente, sem nada a ver com o golpe.
controles = {
    "00.000.000/0001-91": True,    # Banco do Brasil (público, válido)
    "12.ABC.345/01DE-35": True,    # exemplo de CNPJ alfanumérico (formato novo da Receita)
    "11.111.111/1111-11": False,   # todos iguais: inválido por regra
}
for numero, esperado in controles.items():
    ok = cnpj_valido(numero) == esperado
    print(f"   {'✅' if ok else '❌'} controle {numero}: {'válido' if cnpj_valido(numero) else 'inválido'}")

# ── 4. Registrar a pista ──────────────────────────────────────────────────
print("\n📝 Pista registrada:")
print(json.dumps(pista, indent=2, ensure_ascii=False))

### `br/03_ficha_json.py` — Montar a ficha de evidências do caso em JSON, salvar, reler e conferir.

In [ ]:
%cd -q {PASTA_NB}
"""
Montar a ficha de evidências do caso em JSON, salvar, reler e conferir.

Carta 27 (rank B) — Jutsu do Mapa do JSON (parte 3)

O que você aprende:
  • a montar um dicionário com listas e dicionários dentro (estrutura aninhada);
  • json.dump(..., indent=2, ensure_ascii=False): JSON legível e com acentos de verdade;
  • a reler o arquivo com json.load e conferir que nada se perdeu ("ida e volta");
  • o que o JSON NÃO guarda direto (datas, Decimal, conjuntos) e como contornar.

Como rodar (dentro da pasta dia-06-json):
    python br/03_ficha_json.py
"""

import csv
import json
import re
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"

try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))

# ── 1. Juntar o que já sabemos (dias 4, 5 e 6) ────────────────────────────
html = (PASTA_CASO / "vitrine_megaoferta.html").read_text(encoding="utf-8")
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    denuncias = list(csv.DictReader(arquivo, delimiter=";"))

# Conjunto (set) de chaves PIX: tira repetidas. JSON não conhece set → viramos lista ordenada.
chaves_pix = sorted({d["chave_pix_destino"] for d in denuncias})
# Dinheiro: Decimal soma sem erro de centavo. JSON não conhece Decimal → guardamos como texto.
total = sum(Decimal(d["valor"]) for d in denuncias)
# Datas: JSON não conhece date → usamos texto no formato ISO (AAAA-MM-DD).
datas = [date.fromisoformat(d["data"]) for d in denuncias]

ficha = {
    "caso": "Operação Vitrine Fantasma",
    "gerado_em": datetime.now(FUSO_BR).isoformat(timespec="seconds"),
    "loja": {
        "nome": "MegaOferta Brasil",
        "dominio": "megaoferta-brasil.example",
        "cnpj_rodape": re.search(r"CNPJ\s*([\d./-]+)", html).group(1),
        "cnpj_valido": False,                          # conclusão do script 02
        "cep_rodape": re.search(r"CEP\s*(\d{5}-?\d{3})", html).group(1),
        "cep_corresponde_a": "Praça Sete de Setembro, Centro, Belo Horizonte/MG",  # Dia 5
    },
    "denuncias": {
        "quantidade": len(denuncias),
        "primeira": min(datas).isoformat(),
        "ultima": max(datas).isoformat(),
        "valor_total": str(total),                     # "54877.40": texto, sem perder centavos
    },
    "chaves_pix": chaves_pix,
    "evidencias": [
        {"id": "EV-01", "descricao": "Cópia da vitrine", "arquivo": "caso/dados/vitrine_megaoferta.html"},
        {"id": "EV-02", "descricao": "Planilha de denúncias", "arquivo": "caso/dados/denuncias.csv"},
    ],
}

# ── 2. Salvar ─────────────────────────────────────────────────────────────
SAIDAS.mkdir(exist_ok=True)
caminho = SAIDAS / "ficha_evidencias.json"
with open(caminho, "w", encoding="utf-8") as arquivo:
    # indent=2 → uma informação por linha, com recuo (fácil de ler e de comparar no git)
    # ensure_ascii=False → grava "Praça" em vez de "Praça"
    json.dump(ficha, arquivo, indent=2, ensure_ascii=False)
print(f"💾 Ficha salva em {caminho.relative_to(PASTA_DIA)} ({caminho.stat().st_size} bytes)")

# ── 3. Reler e conferir a "ida e volta" ───────────────────────────────────
with open(caminho, encoding="utf-8") as arquivo:
    relida = json.load(arquivo)
print("✅ Releitura idêntica ao original." if relida == ficha else "🚩 Algo mudou na releitura!")

# ── 4. Navegar na ficha relida ────────────────────────────────────────────
print(f"\n🏢 {relida['loja']['nome']} · CNPJ {relida['loja']['cnpj_rodape']} "
      f"({'válido' if relida['loja']['cnpj_valido'] else 'INVÁLIDO'})")
print(f"📨 {relida['denuncias']['quantidade']} denúncias de {relida['denuncias']['primeira']} "
      f"a {relida['denuncias']['ultima']}")
print(f"💰 Total: R$ {Decimal(relida['denuncias']['valor_total'])}  ← de volta a Decimal para fazer conta")
print(f"🔑 {len(relida['chaves_pix'])} chaves PIX distintas:")
for chave in relida["chaves_pix"]:
    print(f"   • {chave}")

# ── 5. O que acontece se tentarmos salvar um tipo que o JSON não conhece? ─
try:
    json.dumps({"data": date(2025, 10, 12)})
except TypeError as erro:
    print(f"\n💡 Lição: {erro}. Por isso convertemos datas para texto ISO antes de salvar.")

## ⚖️ Ética, LGPD e OPSEC

- **CNPJ é público, sócio é gente.** A ficha cadastral de uma empresa é pública por lei, mas os sócios
  pessoas físicas são titulares de dados pela LGPD. Use nome de sócio só quando ele for relevante para
  a investigação, e nunca para expor alguém sem necessidade.
- **Nunca "conserte" o CNPJ do suspeito.** Trocar os dígitos finais até "dar válido" pode apontar para
  uma empresa real e inocente. Número inválido é pista, e a pista é exatamente essa.
- **Validar antes de consultar** economiza cota e reduz rastro: cada consulta registra seu IP no
  servidor da API (e, em sites de terceiros, pode chegar ao conhecimento do alvo).
- **Guarde a origem** de cada dado no próprio JSON (`"fonte": …`, `"registrado_em": …`). Uma ficha sem
  origem não serve como evidência.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que usar dados.get('nome_fantasia', '—') em vez de dados['nome_fantasia']?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque .get é mais rápido", "B) Porque .get não quebra se o campo não existir", "C) Porque colchetes só funcionam com listas", "D) Porque .get converte o valor para texto"]
correta = "B) Porque .get não quebra se o campo não existir"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Com colchetes, um campo ausente gera KeyError e o script para. O .get devolve o valor padrão e o script segue.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. O que faz ensure_ascii=False no json.dump?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Remove os acentos do texto", "B) Grava os acentos como estão ('Praça') em vez de códigos ('Pra\\u00e7a')", "C) Impede que o arquivo seja aberto no Windows", "D) Compacta o JSON numa linha só"]
correta = "B) Grava os acentos como estão ('Praça') em vez de códigos ('Pra\\u00e7a')"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Por padrão o json escapa tudo que não é ASCII. Com ensure_ascii=False (e o arquivo aberto em utf-8), o texto fica legível.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. O CNPJ do rodapé falhou no dígito verificador. Qual é a atitude correta?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Ajustar os dois últimos dígitos e consultar a empresa resultante", "B) Consultar mesmo assim, várias vezes", "C) Registrar como pista (número inventado) e não gastar consulta", "D) Concluir que a loja é de outra cidade"]
correta = "C) Registrar como pista (número inventado) e não gastar consulta"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Um CNPJ com DV inválido não pertence a nenhuma empresa. 'Corrigir' o número pode apontar para uma empresa real e inocente.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 6: Fichário de CNPJs do caso

**Carta 48 · Rank S**

## A missão

O CNPJ do rodapé era falso. A Iara desconfia que **não é o único**: golpista que inventa um número
costuma inventar outros. Ela quer um fichário com todo CNPJ que aparece nos arquivos do caso.

Escreva um script que:

1. Varra os arquivos de `caso/dados/` (`.txt`, `.html`, `.csv`, `.sql`) atrás de CNPJs no formato
   `00.000.000/0000-00`, aceitando também **letras** nas 12 primeiras posições (CNPJ alfanumérico).
2. Guarde **em quais arquivos** cada CNPJ apareceu (um dicionário `cnpj → [arquivos]`).
3. Valide o dígito verificador. **Só consulte a API** para os válidos.
4. Para os válidos, busque na BrasilAPI: razão social, situação cadastral e a lista de sócios (`qsa`).
5. Inclua o CNPJ público de controle `00.000.000/0001-91` para provar que a consulta funciona.
6. Grave `saidas/fichario_cnpjs.json` com `indent=2` e `ensure_ascii=False`, e um resumo
   (`total`, `invalidos`).
7. Funcione **sem internet** usando a amostra de `dados/`.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Nenhuma consulta é feita para CNPJ com DV inválido
- [ ] Os dois CNPJs que o "Kau" usa saem como 🚩, cada um com o arquivo de origem
- [ ] O JSON final abre no navegador ou no VS Code com os acentos certos

## Dica

`dicionario.setdefault(chave, []).append(valor)` cria a lista na primeira vez e só acrescenta nas
seguintes. É o jeito mais curto de montar "um para muitos".

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 6 — Fichário de CNPJs do caso.

Varre os arquivos de texto do caso atrás de CNPJs, valida o dígito verificador
(numérico e alfanumérico), consulta na BrasilAPI só os válidos e grava
saidas/fichario_cnpjs.json com uma ficha por CNPJ e a origem de cada um.
Sem internet (ou com OSINT_OFFLINE=1), usa a amostra do CNPJ público de controle.
"""

import json
import os
import re
import urllib.error
import urllib.request
from pathlib import Path

# CNPJs extras para incluir (ex.: o de controle, público). Separe por vírgula.
CNPJS_EXTRAS = "00.000.000/0001-91"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"
AMOSTRAS = {"00000000000191": PASTA_DIA / "dados" / "brasilapi_cnpj_00000000000191.json"}

# Formato com máscara: 2.3.3/4-2, aceitando letras nas 12 primeiras posições (CNPJ alfanumérico).
PADRAO = re.compile(r"\b[0-9A-Z]{2}\.[0-9A-Z]{3}\.[0-9A-Z]{3}/[0-9A-Z]{4}-\d{2}\b")


def limpar(cnpj):
    return re.sub(r"[^0-9A-Za-z]", "", cnpj).upper()


def cnpj_valido(cnpj):
    c = limpar(cnpj)
    if not re.fullmatch(r"[0-9A-Z]{12}\d{2}", c) or len(set(c)) == 1:
        return False

    def dv(base, pesos):
        resto = sum((ord(ch) - 48) * p for ch, p in zip(base, pesos)) % 11
        return "0" if resto < 2 else str(11 - resto)

    d1 = dv(c[:12], [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])
    return c[12:] == d1 + dv(c[:12] + d1, [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2])


def consultar(cnpj14):
    """Devolve (dados ou None, origem)."""
    if not OFFLINE:
        try:
            req = urllib.request.Request(f"https://brasilapi.com.br/api/cnpj/v1/{cnpj14}",
                                         headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(req, timeout=15) as resposta:
                return json.load(resposta), "ao vivo"
        except urllib.error.HTTPError as erro:
            if erro.code == 404:
                return None, "não encontrado (404)"
        except Exception:
            pass
    if cnpj14 in AMOSTRAS:
        return json.loads(AMOSTRAS[cnpj14].read_text(encoding="utf-8")), "amostra"
    return None, "sem dados"


# ── 1. Varredura: {cnpj: [arquivos onde apareceu]} ────────────────────────
achados = {}
for arquivo in sorted(PASTA_CASO.rglob("*")):
    if arquivo.suffix.lower() in {".txt", ".html", ".csv", ".sql"}:
        for cnpj in PADRAO.findall(arquivo.read_text(encoding="utf-8")):
            achados.setdefault(cnpj, []).append(str(arquivo.relative_to(PASTA_DIA.parent)))
for extra in [c.strip() for c in CNPJS_EXTRAS.split(",") if c.strip()]:
    achados.setdefault(extra, []).append("parâmetro CNPJS_EXTRAS (controle)")

if OFFLINE:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

# ── 2. Uma ficha por CNPJ ─────────────────────────────────────────────────
fichario = []
for cnpj, origens in achados.items():
    ficha = {"cnpj": cnpj, "visto_em": sorted(set(origens)), "valido": cnpj_valido(cnpj),
             "consulta": "não feita (DV inválido)", "razao_social": None, "situacao": None, "socios": []}
    if ficha["valido"]:
        dados, origem = consultar(limpar(cnpj))
        ficha["consulta"] = origem
        if dados:
            ficha["razao_social"] = dados.get("razao_social")
            ficha["situacao"] = dados.get("descricao_situacao_cadastral")
            ficha["socios"] = [{"nome": s.get("nome_socio"), "qualificacao": s.get("qualificacao_socio")}
                               for s in dados.get("qsa", [])]
    fichario.append(ficha)
    marca = "✅" if ficha["valido"] else "🚩"
    print(f"{marca} {cnpj}  {ficha['consulta']:<26} {ficha['razao_social'] or ''}")
    for origem in ficha["visto_em"]:
        print(f"      ↳ {origem}")

# ── 3. Salvar ─────────────────────────────────────────────────────────────
resumo = {"total": len(fichario), "invalidos": sum(not f["valido"] for f in fichario), "fichas": fichario}
SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "fichario_cnpjs.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, indent=2, ensure_ascii=False)
print(f"\n💾 saidas/fichario_cnpjs.json · {resumo['invalidos']} de {resumo['total']} CNPJs são inválidos.")

## 🔎 Inovação OSINT do dia

**CNPJ alfanumérico.** A Receita Federal (Instrução Normativa RFB nº 2.229/2024) definiu que, a partir
de julho de 2026, novas inscrições podem ter letras e números nas 12 primeiras posições. Para OSINT,
isso quebra regex e validadores antigos que só aceitam dígitos: um CNPJ legítimo novo seria
descartado como "inválido". A correção cabe numa linha (`ord(c) - 48`) e já está nos scripts. Biblioteca
Python simples: `re` (nativo) para achar candidatos; a conta é Python puro.

## 📚 Para ir além

- Documentação da rota de CNPJ da BrasilAPI: <https://brasilapi.com.br/docs#tag/CNPJ>
- Módulo `json` do Python (em português): <https://docs.python.org/pt-br/3/library/json.html>
- Notícia (Fenacon) sobre o CNPJ alfanumérico: <https://fenacon.org.br/noticias/cnpj-tera-letras-e-numeros-a-partir-de-julho-de-2026/>
- Página da Receita Federal (busque "CNPJ alfanumérico"): <https://www.gov.br/receitafederal/>
- JSON Path Finder (recomendado pelo autor): <https://jsonpathfinder.com>

## ✅ Checklist

- [ ] Rodei os dois scripts originais e entendi o `- 1` do laço
- [ ] Naveguei pela ficha de CNPJ do Banco do Brasil até a lista de sócios
- [ ] Validei o CNPJ do rodapé **antes** de consultar
- [ ] Salvei e reli a ficha de evidências em JSON
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 6](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)